In [ ]:

import os, sys, glob, shutil, time, json, gc
import numpy as np, pandas as pd
t0=time.perf_counter()
def log(m): print(f"[{time.perf_counter()-t0:6.0f}с] {m}", flush=True)
base=os.path.dirname(glob.glob("/kaggle/input/**/items_human.parquet", recursive=True)[0])
prev=os.path.dirname(glob.glob("/kaggle/input/**/features_human.npy", recursive=True)[0])
os.makedirs("/kaggle/working/src",exist_ok=True)
for p in glob.glob(base+"/*.py"): shutil.copy(p,"/kaggle/working/src/")
open("/kaggle/working/src/__init__.py","a").close()
os.makedirs("/kaggle/working/models",exist_ok=True)
shutil.copy(base+"/anti_words.json","/kaggle/working/models/anti_words.json")
os.chdir("/kaggle/working"); sys.path.insert(0,"/kaggle/working")
from src.hybrid import product_disjoint_pair_masks
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score
from src.attr_features import parse, compare, FEATURE_NAMES
from src.name_features import parse_name, compare_names, build_idf, NAME_FEATURE_NAMES
from src.string_features import compare_strings, STRING_FEATURE_NAMES
from src.neighbour_features import build as nb_build, compare as nb_compare, NEIGHBOUR_FEATURE_NAMES
from src.brand_features import colours, canonical, compare_brands, compare_colours, BRAND_FEATURE_NAMES
from src.dim_features import dimensions, compare_dimensions, compare_translit, DIM_FEATURE_NAMES
from sklearn.feature_extraction.text import TfidfVectorizer
ALL=FEATURE_NAMES+NAME_FEATURE_NAMES+STRING_FEATURE_NAMES+NEIGHBOUR_FEATURE_NAMES+BRAND_FEATURE_NAMES+DIM_FEATURE_NAMES
NB=set(NEIGHBOUR_FEATURE_NAMES); keep=[i for i,n in enumerate(ALL) if n not in NB]

items=pd.read_parquet(base+"/items_human.parquet")
ev=pd.read_parquet(base+"/eval_pairs_mixed.parquet")
y=ev["target"].to_numpy(np.int8); c=ev["category"].astype(str).to_numpy()
def macro(p): return float(np.mean([average_precision_score(y[c==k],p[c==k])
    for k in np.unique(c) if len(np.unique(y[c==k]))>1]))

def featurize(pool, pairs):
    ID=pool["id"].to_numpy(); NAME=pool["name"].astype(str).tolist(); ATTR=pool["attributes"].tolist()
    CAT=pool["category"].astype(str).to_numpy()
    cards={int(i):parse(n,a,name=n) for i,n,a in zip(ID,NAME,ATTR)}
    nm={int(i):parse_name(n) for i,n in zip(ID,NAME)}
    idf,avg=build_idf(list(nm.values()))
    cols={int(i):colours(n+" "+str(a)) for i,n,a in zip(ID,NAME,ATTR)}
    br={int(i):frozenset(x for x in (canonical(v) for v in k.slots.get("brand",())) if x) for i,k in cards.items()}
    dm={int(i):dimensions(n+" "+str(a)) for i,n,a in zip(ID,NAME,ATTR)}
    POS={int(x):r for r,x in enumerate(ID)}; cat_of=dict(zip(ID.tolist(),CAT.tolist()))
    pc=pairs["id1"].map(cat_of).fillna("?").astype(str).to_numpy()
    out=np.zeros((len(pairs),len(ALL)),dtype=np.float32)
    for cat in sorted(set(CAT)):
        rows=np.flatnonzero(pc==cat)
        if not len(rows): continue
        prof=nb_build(pool[["id","name","category"]],categories=[cat])
        g=pool[pool["category"]==cat]
        M=TfidfVectorizer(min_df=1,sublinear_tf=True).fit_transform(g["name"].astype(str).tolist())
        ix={int(x):r for r,x in enumerate(g["id"].to_numpy())}
        for r in rows:
            a,b=int(pairs["id1"].iat[r]),int(pairs["id2"].iat[r])
            if a not in cards or b not in cards: continue
            s=float((M[ix[a]]@M[ix[b]].T).toarray()[0,0]) if (a in ix and b in ix) else 0.0
            d=compare(cards[a],cards[b]); d.update(compare_names(nm[a],nm[b],idf,avg))
            d.update(compare_strings(NAME[POS[a]],NAME[POS[b]])); d.update(nb_compare(a,b,s,prof))
            d.update(compare_brands(br[a],br[b],{})); d.update(compare_colours(cols[a],cols[b]))
            d.update(compare_dimensions(dm[a],dm[b])); d.update(compare_translit(br[a],br[b]))
            out[r]=[d[k] for k in ALL]
        del prof,M; gc.collect()
    return out

# Модели обучаем на готовых признаках (пул обучения не трогаем)
Xh=np.load(prev+"/features_human.npy"); Xl=np.load(prev+"/features_llm.npy")
hm=pd.read_parquet(base+"/matches.parquet",columns=["id1","id2","target"])
lp=pd.read_parquet(base+"/llm_pairs_sel.parquet")
yh=hm["target"].to_numpy(np.int8); yl=lp["label"].to_numpy(np.int8)
tm,vm=product_disjoint_pair_masks(hm["id1"].to_numpy(),hm["id2"].to_numpy(),0,3)
rel=np.flatnonzero(~vm)
P=dict(max_iter=800,learning_rate=0.05,max_leaf_nodes=63,random_state=0,early_stopping=False)
full=HistGradientBoostingClassifier(**P).fit(np.vstack([Xl,Xh[rel]]),np.concatenate([yl,yh[rel]]))
nonb=HistGradientBoostingClassifier(**P).fit(np.vstack([Xl[:,keep],Xh[rel][:,keep]]),
                                             np.concatenate([yl,yh[rel]]))
log("обе модели обучены")

# ГЛАВНОЕ: те же пары, но признаки считаются по РАЗНЫМ пулам товаров.
need=set(pd.unique(np.concatenate([ev["id1"].to_numpy(),ev["id2"].to_numpy()])).tolist())
rng=np.random.default_rng(0)
for tag,pool in (("полный пул", items),
                 ("только товары из пар", items[items["id"].isin(need)]),
                 ("пары + половина прочих",
                  items[items["id"].isin(need) | (rng.random(len(items))<0.5)])):
    E=featurize(pool.reset_index(drop=True), ev)
    a=macro(full.predict_proba(E)[:,1]); b=macro(nonb.predict_proba(E[:,keep])[:,1])
    log(f"{tag:<24} со 128 {a:.6f} | без окрестностей {b:.6f} | пул {len(pool):,}")
log("готово")
